# Stage 3 — Real vs Fake Face Anti-Spoof Classifier
### MobileNetV3-Small + Custom Head | FocalLoss | OneCycleLR
**Architecture:** MobileNetV3-Small → Dropout(0.35) → Linear(512) → Hardswish → BN → Dropout(0.25) → Linear(256) → Hardswish → Dropout(0.15) → Linear(2)


In [ ]:
# Install extra deps
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","gdown","onnx","onnxruntime"],check=True)
print("Deps installed")

In [ ]:
# GPU check and environment detection
import torch, os
IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB  = os.path.exists("/content")
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "/kaggle/input" if IS_KAGGLE else ("./data" if not IS_COLAB else "/content/data")
BATCH     = 64 if (IS_KAGGLE or IS_COLAB) else 32
print(f"Environment: {"Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "Local")}")
print(f"Device: {DEVICE}")
print(f"Data root: {DATA_ROOT}")
print(f"Batch size: {BATCH}")

In [ ]:
# Dataset download (Kaggle input first, fallback to kaggle CLI)
import subprocess, sys, os
datasets = {
    "human_faces":  "kaustubhdhote/human-faces-dataset",
    "fake_140k":    "xhlulu/140k-real-and-fake-faces",
}
for name, slug in datasets.items():
    dest = f"/kaggle/input/{name}" if os.path.exists("/kaggle") else f"./data/{name}"
    if not os.path.exists(dest):
        print(f"Downloading {name}...")
        subprocess.run(["kaggle","datasets","download","-d",slug,"-p","./data","--unzip"],check=False)
    else:
        print(f"{name}: already available at {dest}")

In [ ]:
# Dataset exploration: class distribution + sample grid
import matplotlib.pyplot as plt, numpy as np
from pathlib import Path
from PIL import Image

def show_sample_grid(pos_dir, neg_dir, title="Samples", n=8):
    fig, axes = plt.subplots(2, n, figsize=(n*2, 4))
    for i, d in enumerate([pos_dir, neg_dir]):
        imgs = sorted(Path(d).glob("*.jpg"))[:n] if Path(d).exists() else []
        for j, p in enumerate(imgs):
            axes[i,j].imshow(Image.open(p))
            axes[i,j].axis("off")
    axes[0,0].set_ylabel("Real"); axes[1,0].set_ylabel("Fake")
    plt.suptitle(title); plt.tight_layout(); plt.show()

show_sample_grid(
    f"{DATA_ROOT}/human_faces_dataset/Real_Faces",
    f"{DATA_ROOT}/human_faces_dataset/AI_Generated_Faces"
)

In [ ]:
# Setup Dataset + DataLoader with WeightedRandomSampler
import sys; sys.path.insert(0, ".")
from data.loaders.combined import CombinedAntiSpoofDataset, train_transform, val_transform
from torch.utils.data import DataLoader

train_ds = CombinedAntiSpoofDataset(DATA_ROOT, split="train", transform=train_transform())
val_ds   = CombinedAntiSpoofDataset(DATA_ROOT, split="val",   transform=val_transform())
sampler  = train_ds.make_sampler()

train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH*2, shuffle=False, num_workers=2)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
# Model definition + loss + optimizer
from models.antispoof_net import AntiSpoofNet
from models.losses import CombinedAntiSpoofLoss
import torch

model    = AntiSpoofNet(pretrained=True).to(DEVICE)
criterion= CombinedAntiSpoofLoss()
optimizer= torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler= torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-4,
            steps_per_epoch=len(train_loader), epochs=30, pct_start=0.1, div_factor=25)
print(f"Model params: {model.count_params():,}")

In [ ]:
# Training loop with live plots every 5 epochs
from training.train_classifier import train_one_epoch, validate
from IPython.display import clear_output
import matplotlib.pyplot as plt

EPOCHS = 30
history = {"t_loss":[], "v_auc":[], "v_acer":[]}
best_auc, device = 0.0, torch.device(DEVICE)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE=="cuda")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, scheduler,
                                     type("C",(object,),{"amp":True,"grad_clip":1.0})(), device)
    v_met = validate(model, val_loader, criterion, device)
    history["t_loss"].append(t_loss)
    history["v_auc"].append(v_met["auc"])
    history["v_acer"].append(v_met["acer"])
    if v_met["auc"] > best_auc:
        best_auc = v_met["auc"]
        torch.save({"model_state": model.state_dict()}, "/kaggle/working/best_antispoof.pth")
    if (epoch+1) % 5 == 0:
        clear_output(wait=True)
        fig, (a1,a2) = plt.subplots(1,2,figsize=(12,4))
        a1.plot(history["t_loss"], label="train loss"); a1.set_title("Loss"); a1.legend()
        a2.plot(history["v_auc"],  label="val AUC");  a2.set_title("AUC");  a2.legend()
        plt.tight_layout(); plt.show()
        print(f"Epoch {epoch+1}/{EPOCHS} | AUC={v_met["auc"]:.4f} | ACER={v_met["acer"]:.4f}")

In [ ]:
# Evaluate: confusion matrix, ROC, ACER
from training.evaluate import run_evaluation
metrics = run_evaluation(
    weights_path = "/kaggle/working/best_antispoof.pth",
    dataset_name = "human_faces",
    data_root    = DATA_ROOT,
    output_dir   = "/kaggle/working/eval",
)

In [ ]:
# Export to ONNX + latency benchmark
import torch, time
from models.antispoof_net import AntiSpoofNet

model = AntiSpoofNet(pretrained=False)
ckpt  = torch.load("/kaggle/working/best_antispoof.pth", map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.export_onnx("/kaggle/working/best_antispoof.onnx")
print("ONNX export complete")

dummy = torch.zeros(1, 3, 224, 224)
t0 = time.perf_counter()
for _ in range(100): model(dummy)
print(f"CPU latency: {(time.perf_counter()-t0)*10:.2f} ms/image")

In [ ]:
# FastAPI integration snippet
SNIPPET = """
# FastAPI integration (add to your main attendance system)
import httpx, base64

async def verify_liveness(frame_bytes: bytes, session_id: str) -> dict:
    async with httpx.AsyncClient() as client:
        resp = await client.post(
            "http://localhost:8001/api/v1/liveness/verify",
            files={"frame": ("frame.jpg", frame_bytes, "image/jpeg")},
            data={"session_id": session_id},
        )
    return resp.json()  # {"verdict": "LIVE"|"SPOOF"|"PENDING", "confidence": float, ...}
"""
print(SNIPPET)